# Reproducing every figure and every reported number

This notebook runs inside the supplementary package and needs nothing outside it. No network, no
GPU, no model. Every cell reads the graded records that ship here and recomputes what the paper
reports from them.

It does three things, in order of how much they are worth.

1. **Provenance.** Checks that each judge's predictions are the ones the test seed was drawn
   against, by hash. This is the claim the whole design rests on and it is checkable here.
2. **Figures.** Regenerates every figure in the paper from the records.
3. **Agreement.** Recomputes the numbers the paper prints and compares them to what is printed,
   reporting any disagreement rather than asserting there is none.

Run it top to bottom. Anything that disagrees is reported as `MISMATCH` and the notebook does not
hide it.

In [ ]:
import gzip, hashlib, json, subprocess, sys
from pathlib import Path

# The package root, found from this notebook rather than from any absolute path.
HERE = Path.cwd()
PKG = next((p for p in [HERE, *HERE.parents] if (p / 'experiments').is_dir() and (p / 'MANIFEST.md').exists()), None)
assert PKG is not None, 'run this notebook from inside the supplementary package'
EXP = PKG / 'experiments'
BUILD = PKG / 'paper' / 'iclr2027' / 'build'
print('package root:', PKG)
print('gates present:', sorted(p.name for p in EXP.iterdir() if p.is_dir()))

PROBLEMS = []
def check(label, got, want, tol=5e-4):
    ok = (abs(got - want) <= tol) if isinstance(want, (int, float)) else (got == want)
    if not ok:
        PROBLEMS.append((label, got, want))
    print('%-52s records %-10s paper %-10s %s' % (label, got, want, 'ok' if ok else 'MISMATCH'))
    return ok

## 1. Provenance

The gate's claim is that the test stimuli did not exist when the predictions were written. What is
checkable after the fact is the next best thing: that the predictions graded are byte for byte the
ones pinned when the seed was drawn. `test_seed.json` records each judge's `predictions.json` by
sha256 at the moment of drawing, and the graded record carries the hash it actually used.

In [ ]:
for gate in ('G3e', 'G3f'):
    seed_rec = EXP / gate / 'run_record' / 'test_seed.json'
    verdicts = EXP / gate / 'run_record' / 'verdicts.json'
    if not (seed_rec.exists() and verdicts.exists()):
        print(gate, 'no sealed seed record in this package'); continue
    pinned = json.loads(seed_rec.read_text())['predictions_pinned']
    v = json.loads(verdicts.read_text())
    print('\n%s  test seed %s' % (gate, v.get('test_seed')))
    for judge, rec in v['judges'].items():
        want = pinned.get(judge, {}).get('sha256')
        got = rec.get('predictions_sha256')
        # And recompute the hash from the file that ships, so this is not two records agreeing
        # with each other while both differ from what is here.
        p = EXP / gate / 'run_record' / 'calibration' / (judge + '__served') / 'predictions.json'
        onfile = hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None
        state = 'ok' if (want and got == want and onfile == want) else 'MISMATCH'
        if state != 'ok':
            PROBLEMS.append(('%s/%s provenance' % (gate, judge), got, want))
        print('  %-12s pinned %s  graded %s  on file %s  %s'
              % (judge, (want or '-')[:12], (got or '-')[:12], (onfile or '-')[:12], state))

## 2. Figures

Each script reads the graded records and writes into `paper/iclr2027/figures/`. They are the same
scripts that made the figures in the submitted PDF, unmodified.

In [ ]:
for script, what in (('make_v2_figures.py', 'Figures 1, 2 and 3'),
                     ('channel_fit.py', 'Figure 5 and Table 3, the numeric gate'),
                     ('make_figures.py', 'Figure 4, thresholds against budget')):
    p = BUILD / script
    if not p.exists():
        print('%-22s not in this package' % script); continue
    r = subprocess.run([sys.executable, str(p)], capture_output=True, text=True, cwd=str(BUILD))
    tail = (r.stdout or r.stderr).strip().splitlines()[-2:]
    print('%-22s %-38s rc=%d  %s' % (script, what, r.returncode, ' | '.join(tail)))
    if r.returncode != 0:
        PROBLEMS.append((script, 'rc=%d' % r.returncode, 'rc=0'))

figs = sorted((PKG / 'paper' / 'iclr2027' / 'figures').glob('*.pdf'))
print('\nfigures written:', [f.name for f in figs])

## 3. The numbers the paper prints

Recomputed from the graded records and compared with what the paper states. The paper's values are
written out here so a disagreement is visible rather than silent.

In [ ]:
# --- the worksheet gates, Section 4 -------------------------------------------------
g3c = EXP / 'G3c' / 'run_record'
paper_dev = {'qwen7b__full': 0.078, 'qwen7b__int4': 0.055, 'qwen14b__full': 0.072}
paper_tau = {'qwen7b__full': 0.876, 'qwen7b__int4': 0.908, 'qwen14b__full': 0.948}
for judge in ('qwen7b__full', 'qwen7b__int4', 'qwen14b__full'):
    f = g3c / ('grade_%s.json' % judge)
    if not f.exists():
        print(judge, 'no graded record'); continue
    v = json.loads(f.read_text())['verdicts']
    dev = max(r['max_abs_dev'] for r in v['J1_prediction'].values() if isinstance(r, dict))
    check('G3c %s largest deviation' % judge, round(dev, 3), paper_dev[judge], 1e-3)
    check('G3c %s Kendall tau' % judge, round(v['J3_budget_ordering']['kendall_tau'], 3), paper_tau[judge], 1e-3)

In [ ]:
# --- the served replication, and the judge whose ordering failed ---------------------
f = EXP / 'G3e' / 'run_record' / 'verdicts.json'
if f.exists():
    v = json.loads(f.read_text())
    worst = max(j['J1_max_dev'] for j in v['judges'].values())
    check('G3e largest deviation over its judges', round(worst, 3), 0.077, 1e-3)
    check('G3e gemma31b Kendall tau (the claim that failed)',
          round(v['judges']['gemma31b']['J3_kendall_tau'], 3), 0.804, 1e-3)
    check('G3e gemma31b read-outs at the ladder floor',
          v['judges']['gemma31b']['J3_predicted_at_ladder_floor'], 7)
    check('G3e gate verdict', v['gate'], 'FAIL')

In [ ]:
# --- the second stimulus family, Appendix F and Table 6 ------------------------------
f = EXP / 'G3f' / 'run_record' / 'verdicts.json'
if f.exists():
    v = json.loads(f.read_text())
    want = {'gemma31b': dict(dev=0.066, z=2.97, tau=0.869, floor=2),
            'gemma12b': dict(dev=0.124, z=3.55, tau=0.908, floor=0)}
    for judge, w in want.items():
        j = v['judges'][judge]
        check('G3f %s deviation' % judge, round(j['J1_max_dev'], 3), w['dev'], 1e-3)
        check('G3f %s noise units' % judge, round(j['J1_max_z'], 2), w['z'], 1e-2)
        check('G3f %s Kendall tau' % judge, round(j['J3_kendall_tau'], 3), w['tau'], 1e-3)
        check('G3f %s at the ladder floor' % judge, j['J3_predicted_at_ladder_floor'], w['floor'])
    check('G3f gate verdict', v['gate'], 'PASS')
    # The channel against the nominal rival, the range Table 6 quotes.
    eff = [r['effective'] for j in v['judges'].values() for r in j['J2_sse'].values()]
    nom = [r['nominal'] for j in v['judges'].values() for r in j['J2_sse'].values()]
    print('\nchannel squared error %.4f to %.4f, nominal %.4f to %.4f, channel wins %d of %d cells'
          % (min(eff), max(eff), min(nom), max(nom),
             sum(1 for a, b in zip(eff, nom) if a < b), len(eff)))

In [ ]:
# --- Appendix D, where the predictive power enters -----------------------------------
f = EXP / 'G3c' / 'budget_ladder.json'
if f.exists():
    d = json.loads(f.read_text())
    check('ladder: share closed by the count of scores',
          round(100 * d['share_of_the_gap_closed_by_2_cardinality']), 22, 1)
    check('ladder: share closed by the information rate',
          round(100 * d['share_of_the_gap_closed_by_2b_bits']), 39, 1)
    check('ladder: share closed by the score positions',
          round(100 * d['share_of_the_gap_closed_by_3_codebook']), 22, 1)
    check('ladder: cells with room to explain', d['live_cells'][0], 17)
    check('ladder: cells graded', d['live_cells'][1], 18)
    check('ladder: rate beats the capacity in', d['cells_2b_bits_beats_nominal'][0], 12)
    check('ladder: channel is best in', d['cells_where_the_channel_is_best'][0], 18)

In [ ]:
print('\n' + '=' * 78)
if PROBLEMS:
    print('%d DISAGREEMENT(S) between the records and what the paper prints:' % len(PROBLEMS))
    for label, got, want in PROBLEMS:
        print('  %-56s records %s  paper %s' % (label, got, want))
else:
    print('Every checked number in the paper was reproduced from the records in this package.')
print('=' * 78)

## What this notebook does not do

It does not re-run any judge. Doing that needs the models and, for three of them, a served API, and
the point of shipping the graded records is that a reader does not have to. What it establishes is
that the figures and the reported numbers follow from those records, and that the predictions
graded are the ones the seed was drawn against.

It also does not re-derive the bars. They were fixed by a null simulation before any gate ran, and
their record ships with G3c. A reader who wants to check that the bars were not moved should
compare each gate's `bars` block against G3c's, which every registration here says it takes by
reference.